In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DoubleType,TimestampType,FloatType,DecimalType
import pyspark.sql.functions as F

In [0]:
catalog_name = 'ecommerce'

Cleaning brands Table

In [0]:
df_brands_bronze = spark.table(f'{catalog_name}.bronze.brz_brands')
display(df_brands_bronze.limit(10))

In [0]:
df_brands_silver = df_brands_bronze.withColumn('brand_name',F.trim(F.col("brand_name")))
df_brands_silver.display()

In [0]:
df_brands_silver = df_brands_silver.withColumn('brand_code',F.regexp_replace(F.col('brand_code'),"[^A-Za-z0-9]",""))
df_brands_silver.display()

In [0]:
df_brands_silver.select ("category_code").distinct().display()

In [0]:
category_brands_map=({
    'BOOKS':"BKS",
    'GROCERY':'GRCY',
    'TOYS':'TOY'
})
df_brands_silver = df_brands_silver.replace(category_brands_map,subset=['category_code'])
df_brands_silver.display()

In [0]:
df_brands_silver.select ("category_code").distinct().display()

In [0]:
df_brands_silver.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema',True)\
    .saveAsTable(f'{catalog_name}.silver.sil_brands')


###cleaning category table

In [0]:
df_category_bronze = spark.table(f'{catalog_name}.bronze.brz_category')
df_category_bronze.display()

In [0]:
df_category_silver = df_category_bronze.withColumn('category_code',F.upper(F.col('category_code')))
df_category_silver.display()

In [0]:
df_duplicates = df_category_silver.groupby("category_code").count().filter(F.col('count')>1)
df_duplicates.display()

In [0]:
df_category_silver = df_category_silver.dropDuplicates(['category_code'])
df_category_silver.display()

In [0]:
df_category_silver.write \
    .format('delta')\
    .mode('overwrite')\
    .option('mergeSchema',True)\
    .saveAsTable(f'{catalog_name}.silver.sil_category')


##cleaning products table

In [0]:
df_bronze_products = spark.table(f'{catalog_name}.bronze.brz_products')
df_bronze_products.display()

In [0]:
df_duplicates = df_bronze_products.withColumn('category_code',F.upper(F.col("category_code")))
df_duplicates.display()

In [0]:
df_duplicates = df_duplicates.withColumn('brand_code',F.upper(F.col("brand_code")))
df_duplicates.display()

In [0]:
df_duplicates = df_duplicates.withColumn(
    'length_cm',
    F.regexp_replace(F.col('length_cm'),",",".").cast(DecimalType(10,2)))

    
df_duplicates.display()

In [0]:
df_duplicates = df_duplicates.withColumn(
    'weight_grams',
    F.regexp_replace(F.col("weight_grams"),"g",'').cast(IntegerType())
)
df_duplicates.display()




In [0]:
df_duplicates.printSchema()

In [0]:
df_duplicates.select ('material').distinct().show()

In [0]:
map_material = {
    "Coton": "Cotton",
    "Ruber": "Rubber",
    "Alumium": "Aluminum"
}
df_duplicates = df_duplicates.replace(map_material,subset=["material"])
df_duplicates.show()

In [0]:
df_duplicates.select ('material').distinct().show()

In [0]:
df_products_silver = df_duplicates.withColumn(
    'rating_count',
    F.when(F.col('rating_count').isNotNull(),F.abs(F.col('rating_count')))
    .otherwise(F.lit(0))
)
df_products_silver.show(truncate=True)

In [0]:
df_products_silver.write\
    .format('delta')\
    .mode('overwrite')\
    .option('mergeSchema',True)\
    .saveAsTable(f'{catalog_name}.silver.sil_products')
    

Cleaning Customers table

In [0]:
df_customers_bronze = spark.table(f"{catalog_name}.bronze.brz_customers")
df_customers_bronze.display()

In [0]:
df_customers_bronze.filter(
    F.col('customer_id').isNull()
).count()

In [0]:
cust_dups = df_customers_bronze.dropna(subset=['customer_id'])

In [0]:
cust_dups.filter(F.col('customer_id').isNull()).count()

In [0]:
df_customers_silver = cust_dups.fillna('not available',subset=['phone'])
df_customers_silver.display()

In [0]:
df_customers_silver.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema',True)\
    .saveAsTable(f"{catalog_name}.silver.sil_customers")

In [0]:
df_date_bronze = spark.table(f"{catalog_name}.bronze.brz_date")
df_date_bronze.show()

In [0]:
map_date = {
    'MONDAY':"Monday",
    'monday':'Monday',
    'TUESDAY':"Tuesday",
    'tuesday':'Tuesday',
    'WEDNESDAY':"Wednesday",
    'wednesday':'Wednesday',
    'THURSDAY':"Thursday",
    'thursday':'Thursday',
    'FRIDAY':"Friday",
    'friday':'Friday',
    'SATURDAY':"Saturday",
    'saturday':'Saturday',
    'SUNDAY':"Sunday",
    'sunday':'Sunday'
}
date_dups = df_date_bronze.replace(map_date,subset=['day_name'])
date_dups.show()
                                      

In [0]:
date_dups.filter(F.col('year').isNull()).count()

In [0]:
df_date_silver =date_dups.withColumn(
    'week_of_year',
    F.regexp_replace(F.col("week_of_year"),"-",'')
)
df_date_silver.display()

In [0]:
df_date_silver = df_date_silver.withColumn('date',F.to_date(F.col('date'),'dd-MM-yyyy'))

In [0]:
df_date_silver.display()

In [0]:
df_date_silver.groupBy('date').count().filter('count>1').show()

In [0]:
df_date_silver = df_date_silver.dropDuplicates(subset = ['date'])

In [0]:
df_date_silver.groupBy('date').count().filter('count>1').show()

In [0]:
df_date_silver = df_date_silver.withColumn(
    'quarter',
    F.concat_ws(
        "",
        F.concat(
            F.lit("Q"),
            F.col("quarter"),
            F.lit("-"),
            F.col('year')
        ) 
    )
)

df_date_silver = df_date_silver.withColumn(
    'week_of_year',
    F.concat_ws(
        "-",
        F.concat(
            F.lit("Week"),
            F.col("week_of_year"),
            F.lit("-"),
            F.col('year')
        )
    )
)

In [0]:
df_date_silver.show()

In [0]:
df_date_silver=df_date_silver.withColumnRenamed('week_of_year','week')

In [0]:
df_date_silver.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema','true')\
    .saveAsTable(f'{catalog_name}.silver.sil_date')